<a href="https://colab.research.google.com/github/itsfloo/DM2026-Lab1-Exercise/blob/main/113006261_EECS_%E9%90%98%E9%87%91%E8%8A%B1_%E7%AC%AC%E5%85%AD%E9%80%B1%E4%BD%9C%E6%A5%AD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q "aisuite[groq]"

In [ ]:
import aisuite as ai

In [ ]:
import os
from google.colab import userdata

In [ ]:
api_key = userdata.get('GROQ_API_KEY')
os.environ['GROQ_API_KEY'] = api_key

In [ ]:
model = "groq:openai/gpt-oss-120b"

In [ ]:

import os
from google.colab import userdata
import aisuite as ai

# Recreate the client after loading the key
client = ai.Client()


In [ ]:
system_prompt = ""

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr

In [ ]:
client = ai.Client()

In [ ]:

def clean_message(item):
    if isinstance(item, dict):
        return {
            "role": item.get("role"),
            "content": item.get("content", "")
        }

    if isinstance(item, (tuple, list)) and len(item) == 2:
        return None  # A pair is handled separately below.

    return {
        "role": item.role,
        "content": item.content
    }


def history_to_messages(history):
    messages = []

    for item in (history or []):
        if isinstance(item, (tuple, list)) and len(item) == 2:
            user_message, assistant_message = item

            if user_message:
                messages.append({
                    "role": "user",
                    "content": str(user_message)
                })

            if assistant_message:
                messages.append({
                    "role": "assistant",
                    "content": str(assistant_message)
                })
        else:
            converted = clean_message(item)
            if converted is not None:
                messages.append(converted)

    return messages


In [ ]:
CASES = {
    "Murder Mystery": {
        "title": "The Last Bell",
        "setting": (
            "At 9:00 PM, the owner of Blackwood Manor was found dead "
            "in his locked study. His tea may have been poisoned."
        ),
        "suspects": {
            "Evelyn (niece)": (
                "She claims she was reading in the library from 8:30 to "
                "9:15 PM. She would inherit part of the estate."
            ),
            "Thomas (butler)": (
                "He delivered the victim's tea at 8:45 PM and claims he "
                "never returned to the study."
            ),
            "Victor (business partner)": (
                "He claims he left the manor at 8:40 PM after an argument "
                "about money."
            )
        },
        "clues": {
            "study": (
                "The teacup contains traces of poison. A faint tea stain "
                "appears on a folded service cloth."
            ),
            "kitchen": (
                "The teapot is clean. A small vial is missing from a cabinet "
                "used to store cleaning chemicals."
            ),
            "library": (
                "The library clock stopped at 8:50 PM. This makes Evelyn's "
                "time estimate uncertain."
            ),
            "hallway": (
                "A servant remembers Thomas carrying a folded cloth toward "
                "the study shortly before 9 PM."
            )
        },
        "solution": (
            "Thomas is the intended culprit. He delivered the tea and was "
            "seen carrying a cloth toward the study. These clues support "
            "the theory, but the evidence is circumstantial."
        )
    },

    "Theft Mystery": {
        "title": "The Missing Diamond",
        "setting": (
            "A diamond disappeared from a locked display room at 10 PM "
            "during a private mansion party."
        ),
        "suspects": {
            "Clara (heiress)": (
                "She claims she was talking to guests in the ballroom. "
                "She recently argued with her family about money."
            ),
            "James (security guard)": (
                "He was responsible for the corridor and says a security "
                "camera briefly malfunctioned."
            ),
            "Mia (pianist)": (
                "She performed in the ballroom and denies entering "
                "the display-room corridor."
            )
        },
        "clues": {
            "display room": (
                "The glass case is intact. A thin scratch appears near "
                "its hidden latch."
            ),
            "security room": (
                "The camera recording has a two-minute gap at 9:58 PM. "
                "The footage does not reveal who caused it."
            ),
            "ballroom": (
                "Guests confirm Mia played at 10 PM, but no one confirms "
                "where she was at 9:58 PM."
            ),
            "corridor": (
                "A fresh shoe mark appears near the display-room door. "
                "Its pattern resembles work boots, but it is not conclusive."
            )
        },
        "solution": (
            "James is the intended culprit. The camera gap, his access to "
            "the corridor, and the shoe mark point toward him. However, "
            "the evidence is circumstantial rather than definitive."
        )
    },

    "Missing Person": {
        "title": "The Vanishing Photographer",
        "setting": (
            "Photographer Noah disappeared from a seaside hotel at 7 PM. "
            "His camera and jacket were left in his room."
        ),
        "suspects": {
            "Lena (sister)": (
                "She says she was at the hotel cafe. She had asked Noah "
                "to return home with her."
            ),
            "Oscar (boat operator)": (
                "He claims his boat stayed docked all evening, but he made "
                "a late fuel purchase."
            ),
            "Ruby (receptionist)": (
                "She says Noah asked about the coastal trail before leaving."
            )
        },
        "clues": {
            "hotel room": (
                "Noah's camera contains a photograph taken at 6:42 PM "
                "showing a lighthouse in the distance."
            ),
            "reception": (
                "The guest log shows Noah requested a map of the coastal trail."
            ),
            "harbor": (
                "A fuel receipt records Oscar's purchase at 6:55 PM. "
                "It does not establish where his boat traveled."
            ),
            "coastal trail": (
                "Fresh footprints lead toward an old lighthouse, but "
                "they cannot be reliably linked to any suspect."
            )
        },
        "solution": (
            "The intended explanation is that Noah went toward the old "
            "lighthouse. The evidence does not prove that a suspect harmed "
            "him, so a careful detective should avoid unsupported accusations."
        )
    },

    "Strange Mystery": {
        "title": "The Clock That Rang at Midnight",
        "setting": (
            "At an abandoned hotel, a clock rang at midnight even though "
            "its mechanism was believed to have been removed."
        ),
        "suspects": {
            "Mr. Vale (caretaker)": (
                "He has keys to the hotel and claims he was in the basement."
            ),
            "Iris (urban explorer)": (
                "She entered to take photographs and claims she stayed "
                "near the lobby."
            ),
            "Leo (neighbor)": (
                "He says he heard the bell from outside and never entered."
            )
        },
        "clues": {
            "clock": (
                "The clock hands are frozen at 11:50, but the bell mechanism "
                "has a fresh scratch."
            ),
            "basement": (
                "A thin wire runs from a storage shelf toward the ceiling."
            ),
            "lobby": (
                "A trail in the dust leads behind the clock cabinet."
            ),
            "entrance": (
                "A muddy footprint appears inside the entrance, but it "
                "cannot identify who entered."
            )
        },
        "solution": (
            "A hidden wire may have been used to trigger the bell. "
            "The caretaker had access, but the clues do not conclusively "
            "identify who set it up."
        )
    }
}


In [ ]:

DIFFICULTIES = {
    "Easy": (
        "Give clear clues and helpful guidance. Explain how clues connect."
    ),
    "Medium": (
        "Require the player to connect clues and consider alternative "
        "explanations before making an accusation."
    ),
    "Hard": (
        "Encourage close examination of timelines, alibis, and contradictions. "
        "Do not invent evidence that is absent from the case file."
    )
}

GAME_RULES = """
You are Mystery Master, the narrator of an interactive detective game.

Speak in English and create an immersive detective experience.

RULES:
1. Use only the supplied case facts, suspects, and clues.
2. Keep the intended solution and established evidence consistent.
3. Never reveal the solution before the player asks for it or makes
   a final accusation.
4. If the player investigates a location, reveal its corresponding clue.
5. If the player asks about a suspect, explain their established alibi.
6. After each reply, offer 2-4 numbered actions to choose from.
7. Allow custom actions and follow-up questions.
8. When the player makes an accusation, assess their reasoning and
   explain the intended solution fairly.
9. Do not claim the evidence proves guilt when it is only circumstantial.
10. Keep responses concise, clear, and engaging.
"""

In [ ]:

def detective_chat(message, history, case_type, difficulty):

    if not message or not message.strip():
        return "Choose an investigation action to continue."

    case = CASES[case_type]

    suspects_text = "\n".join(
        f"- {name}: {details}"
        for name, details in case["suspects"].items()
    )

    clues_text = "\n".join(
        f"- {location}: {clue}"
        for location, clue in case["clues"].items()
    )

    case_context = f"""
CASE: {case['title']}
TYPE: {case_type}
DIFFICULTY: {difficulty}

SETTING:
{case['setting']}

SUSPECTS:
{suspects_text}

AVAILABLE LOCATION CLUES:
{clues_text}

INTENDED SOLUTION — KEEP SECRET UNTIL THE PLAYER SOLVES THE CASE:
{case['solution']}

DIFFICULTY INSTRUCTIONS:
{DIFFICULTIES[difficulty]}
"""

    messages = [
        {
            "role": "system",
            "content": GAME_RULES + "\n\n" + case_context
        }
    ]

    for item in (history or []):
        messages.append(clean_message(item))

    messages.append({
        "role": "user",
        "content": message.strip()
    })

    response = client.chat.completions.create(
        model=model,
        messages=messages
    )

    return response.choices[0].message.content or (
        "The investigation continues, Detective. What would you like to do next?"
    )



In [ ]:

# Build the interactive Mystery Master interface.
with gr.Blocks(title="Mystery Master - Detective Game") as detective_app:

    gr.Markdown("""
    # 🔎 Mystery Master
    ### Every clue matters. Every suspect has a story.

    Choose your case and difficulty, then investigate!
    """)

    with gr.Row():
        case_choice = gr.Dropdown(
            choices=list(CASES.keys()),
            value="Theft Mystery",
            label="Choose Your Case"
        )

        difficulty_choice = gr.Dropdown(
            choices=list(DIFFICULTIES.keys()),
            value="Medium",
            label="Choose Difficulty"
        )

    gr.Markdown("""
    **Try these actions:**
    - Inspect the display room
    - Interrogate James
    - Check the timeline
    - Make an accusation
    """)

    chatbot = gr.Chatbot(height=420, label="Case Investigation")

    msg = gr.Textbox(
        placeholder="What would you like to investigate, Detective?",
        label="Your Action"
    )

    with gr.Row():
        send_btn = gr.Button("Investigate", variant="primary")
        clear_btn = gr.Button("New Investigation")

    # Process the player's message and update the displayed chat.
    def respond(message, history, case_type, difficulty):
        history = history or []

        if not message or not message.strip():
            return "", history

        answer = detective_chat(
            message,
            history,
            case_type,
            difficulty
        )

        updated_history = history + [(message, answer)]

        return "", updated_history

    send_btn.click(
        fn=respond,
        inputs=[msg, chatbot, case_choice, difficulty_choice],
        outputs=[msg, chatbot]
    )

    msg.submit(
        fn=respond,
        inputs=[msg, chatbot, case_choice, difficulty_choice],
        outputs=[msg, chatbot]
    )

    clear_btn.click(
        fn=lambda: ("", []),
        inputs=None,
        outputs=[msg, chatbot]
    )

# Launch the web application.
detective_app.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8ee04e22c881954604.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
demo.launch(share=True, debug=True)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ad4fbeaf5aabd60063.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
